# Text Extraction with BERT

**Author:** [Apoorv Nandan](https://twitter.com/NandanApoorv)<br>
**Date created:** 2020/05/23<br>
**Last modified:** 2020/05/23<br>
**Description:** Fine tune pretrained BERT from HuggingFace Transformers on SQuAD.

## Introduction

This demonstration uses SQuAD (Stanford Question-Answering Dataset).
In SQuAD, an input consists of a question, and a paragraph for context.
The goal is to find the span of text in the paragraph that answers the question.
We evaluate our performance on this data with the "Exact Match" metric,
which measures the percentage of predictions that exactly match any one of the
ground-truth answers.

We fine-tune a BERT model to perform this task as follows:

1. Feed the context and the question as inputs to BERT.
2. Take two vectors S and T with dimensions equal to that of
   hidden states in BERT.
3. Compute the probability of each token being the start and end of
   the answer span. The probability of a token being the start of
   the answer is given by a dot product between S and the representation
   of the token in the last layer of BERT, followed by a softmax over all tokens.
   The probability of a token being the end of the answer is computed
   similarly with the vector T.
4. Fine-tune BERT and learn S and T along the way.

**References:**

- [BERT](https://arxiv.org/abs/1810.04805)
- [SQuAD](https://arxiv.org/abs/1606.05250)

## Setup

In [2]:
import os
import re
import json
import string
import numpy as np
from tokenizers import BertWordPieceTokenizer
from transformers import BertTokenizer, BertConfig

max_len = 384
configuration = BertConfig()  # default parameters and configuration for BERT

In [3]:
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from transformers import BertModel # This will now import the PyTorch version

## Set-up BERT tokenizer

In [4]:
# Save the slow pretrained tokenizer
slow_tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
save_path = "bert_base_uncased/"
if not os.path.exists(save_path):
    os.makedirs(save_path)
slow_tokenizer.save_pretrained(save_path)

# Load the fast tokenizer from saved file
tokenizer = BertWordPieceTokenizer("bert_base_uncased/vocab.txt", lowercase=True)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


## Load the data

In [5]:
# Download the SQuAD datasets
!wget https://rajpurkar.github.io/SQuAD-explorer/dataset/train-v1.1.json -O train.json
!wget https://rajpurkar.github.io/SQuAD-explorer/dataset/dev-v1.1.json -O eval.json

# Define the file paths for the next cell
train_path = "train.json"
eval_path = "eval.json"

--2025-11-14 23:42:27--  https://rajpurkar.github.io/SQuAD-explorer/dataset/train-v1.1.json
Resolving rajpurkar.github.io (rajpurkar.github.io)... 185.199.108.153, 185.199.109.153, 185.199.110.153, ...
Connecting to rajpurkar.github.io (rajpurkar.github.io)|185.199.108.153|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 30288272 (29M) [application/json]
Saving to: ‘train.json’

train.json          100%[===================>]  28.88M   184MB/s    in 0.2s    

2025-11-14 23:42:27 (184 MB/s) - ‘train.json’ saved [30288272/30288272]

--2025-11-14 23:42:28--  https://rajpurkar.github.io/SQuAD-explorer/dataset/dev-v1.1.json
Resolving rajpurkar.github.io (rajpurkar.github.io)... 185.199.108.153, 185.199.109.153, 185.199.110.153, ...
Connecting to rajpurkar.github.io (rajpurkar.github.io)|185.199.108.153|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 4854279 (4.6M) [application/json]
Saving to: ‘eval.json’

eval.json           100%[==========

## Preprocess the data

1. Go through the JSON file and store every record as a `SquadExample` object.
2. Go through each `SquadExample` and create `x_train, y_train, x_eval, y_eval`.

In [6]:

class SquadExample:
    def __init__(self, question, context, start_char_idx, answer_text, all_answers):
        self.question = question
        self.context = context
        self.start_char_idx = start_char_idx
        self.answer_text = answer_text
        self.all_answers = all_answers
        self.skip = False

    def preprocess(self):
        context = self.context
        question = self.question
        answer_text = self.answer_text
        start_char_idx = self.start_char_idx

        # Clean context, answer and question
        context = " ".join(str(context).split())
        question = " ".join(str(question).split())
        answer = " ".join(str(answer_text).split())

        # Find end character index of answer in context
        end_char_idx = start_char_idx + len(answer)
        if end_char_idx >= len(context):
            self.skip = True
            return

        # Mark the character indexes in context that are in answer
        is_char_in_ans = [0] * len(context)
        for idx in range(start_char_idx, end_char_idx):
            is_char_in_ans[idx] = 1

        # Tokenize context
        tokenized_context = tokenizer.encode(context)

        # Find tokens that were created from answer characters
        ans_token_idx = []
        for idx, (start, end) in enumerate(tokenized_context.offsets):
            if sum(is_char_in_ans[start:end]) > 0:
                ans_token_idx.append(idx)

        if len(ans_token_idx) == 0:
            self.skip = True
            return

        # Find start and end token index for tokens from answer
        start_token_idx = ans_token_idx[0]
        end_token_idx = ans_token_idx[-1]

        # Tokenize question
        tokenized_question = tokenizer.encode(question)

        # Create inputs
        input_ids = tokenized_context.ids + tokenized_question.ids[1:]
        token_type_ids = [0] * len(tokenized_context.ids) + [1] * len(
            tokenized_question.ids[1:]
        )
        attention_mask = [1] * len(input_ids)

        # Pad and create attention masks.
        # Skip if truncation is needed
        padding_length = max_len - len(input_ids)
        if padding_length > 0:  # pad
            input_ids = input_ids + ([0] * padding_length)
            attention_mask = attention_mask + ([0] * padding_length)
            token_type_ids = token_type_ids + ([0] * padding_length)
        elif padding_length < 0:  # skip
            self.skip = True
            return

        self.input_ids = input_ids
        self.token_type_ids = token_type_ids
        self.attention_mask = attention_mask
        self.start_token_idx = start_token_idx
        self.end_token_idx = end_token_idx
        self.context_token_to_char = tokenized_context.offsets


with open(train_path) as f:
    raw_train_data = json.load(f)

with open(eval_path) as f:
    raw_eval_data = json.load(f)


def create_squad_examples(raw_data):
    squad_examples = []
    for item in raw_data["data"]:
        for para in item["paragraphs"]:
            context = para["context"]
            for qa in para["qas"]:
                question = qa["question"]
                answer_text = qa["answers"][0]["text"]
                all_answers = [_["text"] for _ in qa["answers"]]
                start_char_idx = qa["answers"][0]["answer_start"]
                squad_eg = SquadExample(
                    question, context, start_char_idx, answer_text, all_answers
                )
                squad_eg.preprocess()
                squad_examples.append(squad_eg)
    return squad_examples


def create_inputs_targets(squad_examples):
    dataset_dict = {
        "input_ids": [],
        "token_type_ids": [],
        "attention_mask": [],
        "start_token_idx": [],
        "end_token_idx": [],
    }
    for item in squad_examples:
        if item.skip == False:
            for key in dataset_dict:
                dataset_dict[key].append(getattr(item, key))
    for key in dataset_dict:
        dataset_dict[key] = np.array(dataset_dict[key])

    x = [
        dataset_dict["input_ids"],
        dataset_dict["token_type_ids"],
        dataset_dict["attention_mask"],
    ]
    y = [dataset_dict["start_token_idx"], dataset_dict["end_token_idx"]]
    return x, y


train_squad_examples = create_squad_examples(raw_train_data)
x_train, y_train = create_inputs_targets(train_squad_examples)
print(f"{len(train_squad_examples)} training points created.")

eval_squad_examples = create_squad_examples(raw_eval_data)
x_eval, y_eval = create_inputs_targets(eval_squad_examples)
print(f"{len(eval_squad_examples)} evaluation points created.")

87599 training points created.
10570 evaluation points created.


Create the Question-Answering Model using BERT and Functional API

In [7]:
class QAModel(nn.Module):
    """
    This is the PyTorch equivalent of your create_model() function.
    """
    def __init__(self):
        super(QAModel, self).__init__()

        # 1. BERT encoder (loads the PyTorch version by default)
        self.bert = BertModel.from_pretrained("bert-base-uncased")

        # Get BERT's hidden size (usually 768) for the output layers
        hidden_size = self.bert.config.hidden_size

        # 2. Output layers (equivalent to your Keras Dense(1) layers)
        # We want to predict a logit for the start and end of the answer span
        self.qa_outputs = nn.Linear(hidden_size, 2) # 2 outputs: start logit, end logit

    # The forward pass is where the data flows through the model
    def forward(self, input_ids, token_type_ids, attention_mask):

        # 1. Get BERT embeddings
        # The output is a tuple; we want the last_hidden_state
        bert_output = self.bert(
            input_ids,
            token_type_ids=token_type_ids,
            attention_mask=attention_mask
        )
        # sequence_output has shape [batch_size, max_len, hidden_size]
        sequence_output = bert_output.last_hidden_state

        # 2. Pass the embeddings through the final linear layer
        # This gets us logits for start and end, shape [batch_size, max_len, 2]
        logits = self.qa_outputs(sequence_output)

        # 3. Separate the start and end logits
        # We split the last dimension (size 2) into two separate tensors
        start_logits, end_logits = logits.split(1, dim=-1)

        # 4. Squeeze the last dimension to get [batch_size, max_len]
        start_logits = start_logits.squeeze(-1)
        end_logits = end_logits.squeeze(-1)

        return start_logits, end_logits

This code should preferably be run on Google Colab TPU runtime.
With Colab TPUs, each epoch will take 5-6 minutes.

In [8]:
# --- Model and Device Setup ---
# Check if GPU is available and set the device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Instantiate the model and move it to the GPU
model = QAModel().to(device)

# --- Optimizer ---
# Adam optimizer, just like in your Keras code
optimizer = torch.optim.Adam(model.parameters(), lr=5e-5)

# --- Loss Function ---
# CrossEntropyLoss is the equivalent of SparseCategoricalCrossentropy(from_logits=True)
# It expects raw logits from the model and integer labels.
loss_fn = nn.CrossEntropyLoss()

# --- Convert NumPy Data to PyTorch Tensors ---
# Your x_train is a list of 3 numpy arrays, and y_train is a list of 2
# Let's convert them to tensors.

# x_train
x_train_ids = torch.tensor(x_train[0], dtype=torch.long)
x_train_token_types = torch.tensor(x_train[1], dtype=torch.long)
x_train_mask = torch.tensor(x_train[2], dtype=torch.long)

# y_train (our labels)
y_train_start = torch.tensor(y_train[0], dtype=torch.long)
y_train_end = torch.tensor(y_train[1], dtype=torch.long)

# --- Create DataLoader ---
# This will handle batching our data for us
batch_size = 16 # Same as in your model.fit() call

# TensorDataset bundles all our tensors together
train_dataset = TensorDataset(
    x_train_ids,
    x_train_mask,
    x_train_token_types,
    y_train_start,
    y_train_end
)

# DataLoader creates an iterator that gives us batches
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

Using device: cpu


## Create evaluation Callback

This callback will compute the exact match score using the validation data
after every epoch.

In [9]:
# --- Convert Eval Data to Tensors ---
x_eval_ids = torch.tensor(x_eval[0], dtype=torch.long)
x_eval_token_types = torch.tensor(x_eval[1], dtype=torch.long)
x_eval_mask = torch.tensor(x_eval[2], dtype=torch.long)
y_eval_start = torch.tensor(y_eval[0], dtype=torch.long)
y_eval_end = torch.tensor(y_eval[1], dtype=torch.long)

# --- Create Eval DataLoader ---
# Use the same batch_size you set for training
eval_dataset = TensorDataset(
    x_eval_ids,
    x_eval_mask,
    x_eval_token_types
)

# No shuffle needed for evaluation
eval_loader = DataLoader(eval_dataset, batch_size=batch_size)

In [10]:
def normalize_text(text):
    text = text.lower()
    # Remove punctuations
    exclude = set(string.punctuation)
    text = "".join(ch for ch in text if ch not in exclude)
    # Remove articles
    regex = re.compile(r"\b(a|an|the)\b", re.UNICODE)
    text = re.sub(regex, " ", text)
    # Remove extra white space
    text = " ".join(text.split())
    return text

def evaluate(model, eval_loader, eval_squad_examples):
    print("\nEvaluating...")
    # Set model to evaluation mode (disables dropout, etc.)
    model.eval()

    all_pred_starts = []
    all_pred_ends = []

    # No gradients needed for evaluation
    with torch.no_grad():
        for batch in eval_loader:
            # Move batch to GPU
            batch = [item.to(device) for item in batch]
            # Unpack the batch (assuming no labels in loader)
            b_input_ids, b_attn_mask, b_token_types = batch

            # Get model predictions (logits)
            start_logits, end_logits = model(
                input_ids=b_input_ids,
                attention_mask=b_attn_mask,
                token_type_ids=b_token_types
            )

            # Move logits to CPU and convert to numpy
            all_pred_starts.append(start_logits.detach().cpu().numpy())
            all_pred_ends.append(end_logits.detach().cpu().numpy())

    # Concatenate all batch predictions
    pred_start = np.concatenate(all_pred_starts, axis=0)
    pred_end = np.concatenate(all_pred_ends, axis=0)

    # --- Now we run the same Exact Match logic as before ---
    count = 0
    # Get the same list of non-skipped examples
    eval_examples_no_skip = [_ for _ in eval_squad_examples if _.skip == False]

    for idx, (start, end) in enumerate(zip(pred_start, pred_end)):
        squad_eg = eval_examples_no_skip[idx]
        offsets = squad_eg.context_token_to_char

        # Get the token index with the highest logit
        start = np.argmax(start)
        end = np.argmax(end)

        if start >= len(offsets):
            continue

        pred_char_start = offsets[start][0]
        if end < len(offsets):
            pred_char_end = offsets[end][1]
            pred_ans = squad_eg.context[pred_char_start:pred_char_end]
        else:
            pred_ans = squad_eg.context[pred_char_start:]

        normalized_pred_ans = normalize_text(pred_ans)
        normalized_true_ans = [normalize_text(_) for _ in squad_eg.all_answers]
        if normalized_pred_ans in normalized_true_ans:
            count += 1

    # y_eval[0] is the original numpy array of start positions
    acc = count / len(y_eval[0])
    print(f"\n--- Exact Match Score: {acc:.3f} ---")

    # Set model back to train mode
    model.train()
    return acc

## Train and Evaluate

In [11]:
# exact_match_callback = ExactMatch(x_eval, y_eval)
# model.fit(
#     x_train,
#     y_train,
#     epochs=1,  # For demonstration, 3 epochs are recommended
#     verbose=2,
#     batch_size=64,
#     callbacks=[exact_match_callback],
# )

In [ ]:
# Set the model to training mode
model.train()

# Define number of epochs
epochs = 1 # Same as in your model.fit() call

for epoch in range(epochs):
    print(f"\n--- Epoch {epoch+1} ---")

    # Loop over each batch from the DataLoader
    for batch_num, batch in enumerate(train_loader):

        # 1. Move the batch to the GPU
        batch = [item.to(device) for item in batch]
        b_input_ids, b_attn_mask, b_token_types, b_start_labels, b_end_labels = batch

        # 2. Clear previous gradients
        optimizer.zero_grad()

        # 3. Forward pass: get model outputs (logits)
        start_logits, end_logits = model(
            input_ids=b_input_ids,
            attention_mask=b_attn_mask,
            token_type_ids=b_token_types
        )

        # 4. Calculate the loss
        # We calculate the loss for the start and end positions separately
        start_loss = loss_fn(start_logits, b_start_labels)
        end_loss = loss_fn(end_logits, b_end_labels)
        # Total loss is the average of the two
        total_loss = (start_loss + end_loss) / 2

        # 5. Backward pass: compute gradients
        total_loss.backward()

        # 6. Update weights
        optimizer.step()

        if (batch_num + 1) % 100 == 0:
            print(f"  Batch {batch_num+1}/{len(train_loader)} - Loss: {total_loss.item():.4f}")

print("\nTraining complete!")


--- Epoch 1 ---


In [ ]:
# --- Run Evaluation ---
evaluate(model, eval_loader, eval_squad_examples)